# Создание моделей для предсказания свойств углепластика, полученного по вакуумной технологии

In [48]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import f_classif

import matplotlib.pyplot as plt 

# инструменты для построения модели:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # инструмент для создания и обучения модели
from sklearn.ensemble import RandomForestRegressor # инструмент для создания и обучения модели
from sklearn import metrics # инструменты для оценки точности модели
from xgboost import XGBRegressor

import pickle

RANDOM_SEED = 42

In [49]:
df = pd.read_csv('data/dataset_prepaired.csv')
df.head()

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,technology,Thickness of the monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
0,188.0,1.758,4.59,253.0,1.814229,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
1,189.0,1.758,4.48,260.0,1.723077,1.1,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
2,188.0,1.759,4.28,257.0,1.665370,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
3,187.0,1.758,4.77,256.0,1.863281,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
4,190.0,1.757,4.56,255.0,1.788235,0.9,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0


In [50]:
df = df[df['technology'] == 0]
df = df.drop('technology', axis=1)
#df_autoclave = df[df['technology'] == 1]

In [51]:
df = df.rename(columns={'Thickness of the monolayer' : 'Thickness_monolayer',
                        'Module_plastik ' : 'Module_plastik'})

### Модель для предсказания толщины монослоя Thickness_monolayer

In [52]:
train_data_thickness = df.drop(['density', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_thickness.drop(['Thickness_monolayer'], axis=1))
y = np.array(train_data_thickness.Thickness_monolayer.values)

In [53]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [54]:
# НАСТРОЙКИ 
model_rf_thikness = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

In [55]:
# обучаем модель на тестовом наборе данных
model_rf_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_thikness.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [56]:
def mean_absolute_percentage_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

In [57]:
# сравниваем предсказанные значения (y_pred) с реальными (y_test), 
# метрика mean squared error, MSE показывает среднеквадратичное отклонение:

def mean_squared_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr)**2)) 

In [58]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.003


In [59]:
# save model
with open('vac_model_rf_thikness.pkl','wb') as f:
    pickle.dump(model_rf_thikness,f)

In [60]:
model_lr_thikness = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_thikness.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.607


In [ ]:
# save
with open('vac_model_lr_thikness.pkl','wb') as f:
    pickle.dump(model_lr_thikness,f)

### Модель для предсказания плотности углепластика density

In [62]:
train_data_density = df.drop(['Thickness_monolayer', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_density.drop(['density'], axis=1))
y = np.array(train_data_density.density.values)

In [63]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_density = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_density.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [64]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.002


In [65]:
# save
with open('vac_model_rf_density.pkl','wb') as f:
    pickle.dump(model_rf_density,f)

In [66]:
model_lr_density = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_density.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.236


In [67]:
# save
with open('vac_model_lr_density.pkl','wb') as f:
    pickle.dump(model_lr_density,f)

### Модель для предсказания прочности углепластика Strength

In [68]:
train_data_strength = df.drop(['Thickness_monolayer', 'density', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_strength.drop(['Strength_plastik'], axis=1))
y = np.array(train_data_strength.Strength_plastik.values)

In [69]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_strength = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_strength.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [70]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 8.635
MAPE: 0.018


In [71]:
# save
with open('vac_model_rf_strength.pkl','wb') as f:
    pickle.dump(model_rf_strength,f)

In [72]:
model_lr_strength = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_strength.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 49.333
MAPE: 0.609


In [73]:
# save
with open('vac_model_lr_strength.pkl','wb') as f:
    pickle.dump(model_lr_strength,f)

In [74]:
model_xgb_strenght = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_strenght.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_strenght.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 3.63
MAPE: 0.014


/home/alexandr/anaconda3/envs/ML/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:43:45] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [75]:
# save
with open('vac_model_xgb_strenght.pkl','wb') as f:
    pickle.dump(model_xgb_strenght,f)

### Модель для предсказания модуля углепластика Module

In [76]:
train_data_module = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_module.drop(['Module_plastik'], axis=1))
y = np.array(train_data_module.Module_plastik .values)

In [77]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_module = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_module.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [78]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.002
MAPE: 0.003


In [79]:
# save
with open('vac_model_rf_module.pkl','wb') as f:
    pickle.dump(model_rf_module,f)

In [80]:
model_lr_module = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_module.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 1.077
MAPE: 1.127


In [81]:
# save
with open('vac_model_lr_module.pkl','wb') as f:
    pickle.dump(model_lr_module,f)

### Модель для предсказания межслоевой прочности углепластика LSS

In [82]:
train_data_lss = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik',
                             'Plastik_Tg'], axis=1)

X = np.array(train_data_lss.drop(['LSS'], axis=1))
y = np.array(train_data_lss.LSS.values)

In [83]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_lss = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_lss.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [84]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.025
MAPE: 0.01


In [85]:
# save
with open('vac_model_rf_lss.pkl','wb') as f:
    pickle.dump(model_rf_lss,f)

In [30]:
model_lr_lss = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_lss.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 1.779
MAPE: 1.388


In [86]:
# save
with open('vac_model_lr_lss.pkl','wb') as f:
    pickle.dump(model_lr_lss,f)

# Модель для предсказания температуры стеклования углепластика Tg

In [87]:
train_data_Tg = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik', 
                         'LSS'], axis=1)

X = np.array(train_data_Tg.drop(['Plastik_Tg'], axis=1))
y = np.array(train_data_Tg.Plastik_Tg.values)

In [88]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_Tg = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_Tg.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [89]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.021
MAPE: 0.006


In [90]:
# save
with open('vac_model_rf_Tg.pkl','wb') as f:
    pickle.dump(model_rf_Tg,f)

In [91]:
model_lr_Tg = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_Tg.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.252
MAPE: 0.213


In [92]:
# save
with open('vac_model_lr_Tg.pkl','wb') as f:
    pickle.dump(model_lr_Tg,f)